# 20. ResidualverbindungenAus *Transformer von Grund auf*.

In [ ]:
# In Colab liegt nichts auf der Festplatte, also holt diese Zelle# die zwei oder drei Dateien, die das Kapitel braucht. Wer das# Repository geklont hat, findet sie schon vor, und es wird# nichts heruntergeladen.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/notebooks/de/code/"PLACES = [".", "code", "../code"]for name in ["ch20_residuals.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

Was eine Residualverbindung mit einem Gradienten macht.Ein tiefer Stapel von Schichten, zweimal gebaut: einmal schlicht und einmal so, dass die Eingabe jeder Schicht zu ihrer Ausgabe addiert wird. An beiden wird dasselbe gemessen, nämlich wie viel Gradient jede Tiefe erreicht.

In [ ]:
import numpy as npimport torchimport torch.nn as nn

Ein Turm gleicher Schichten, mit oder ohne Umgehung.

In [ ]:
class Stack(nn.Module):    def __init__(self, depth, width, residual, seed=0):        super().__init__()        torch.manual_seed(seed)        self.layers = nn.ModuleList(            nn.Linear(width, width) for _ in range(depth))        self.residual = residual    def forward(self, x):        for layer in self.layers:            out = torch.relu(layer(x))            x = x + out if self.residual else out        return x

Einen Batch durchschicken, dann überall den Gradienten messen.

In [ ]:
def gradient_by_depth(depth=40, width=64, residual=True, seed=0):    torch.manual_seed(seed)    model = Stack(depth, width, residual, seed)    x = torch.randn(16, width)    loss = model(x).pow(2).mean()    loss.backward()    return [float(layer.weight.grad.abs().mean())            for layer in model.layers]

Wie groß die Werte selbst auf dem Weg nach oben werden.

In [ ]:
def activation_size(depth=40, width=64, residual=True, seed=0):    torch.manual_seed(seed)    model = Stack(depth, width, residual, seed)    x = torch.randn(16, width)    sizes = []    with torch.no_grad():        for layer in model.layers:            out = torch.relu(layer(x))            x = x + out if residual else out            sizes.append(float(x.abs().mean()))    return sizes

In [ ]:
def main():    depth = 40    with_res = gradient_by_depth(depth, residual=True)    without = gradient_by_depth(depth, residual=False)    print(f"MITTLERER GRADIENT, DER JEDE SCHICHT ERREICHT, TIEFE {depth}")    print("  Schicht mit Residuen      ohne")    for i in [0, 4, 9, 19, 29, depth - 1]:        print(f"  {i:<7} {with_res[i]:<16.3e} {without[i]:.3e}")    print("\nWIE TIEF DER GRADIENT VON OBEN NACH UNTEN FÄLLT")    runs = [("mit Residuen", with_res), ("ohne", without)]    for name, values in runs:        ratio = values[-1] / max(values[0], 1e-30)        print(f"  {name:<16} die oberste Schicht sieht {ratio:.1e}-mal")        print(f"  {'':<16} so viel wie die unterste")    print("\nWARUM, IN EINER ZEILE ARITHMETIK")    print("  ohne Residuum wird der Gradient auf dem Rückweg mit der")    print("  Gewichtsmatrix jeder Schicht multipliziert, die Faktoren")    print("  schaukeln sich auf. Mit Residuum ist die Ableitung von")    print("  x + f(x) gleich 1 + f', also gibt es immer einen Rückweg,")    print("  der mit eins multipliziert")    print("\nDERSELBE STAPEL, VIEL TIEFER")    print("  Tiefe   mit Residuen     ohne")    for d in [10, 40, 100]:        a = gradient_by_depth(d, residual=True)        b = gradient_by_depth(d, residual=False)        print(f"  {d:<7} {a[0]:<16.3e} {b[0]:.3e}")    print("  der Gradient erreicht jetzt den Boden, und er ist unterwegs")    print("  riesig geworden. Residuen haben ein Problem gelöst und")    print("  ein anderes geschaffen")    print("\nWARUM ER WUCHS, GEMESSEN AUF DEM WEG NACH OBEN")    print("  Schicht Größe der Aktivierung mit Residuen ohne")    grew = activation_size(40, residual=True)    flat = activation_size(40, residual=False)    for i in [0, 9, 19, 29, 39]:        print(f"  {i:<7} {grew[i]:<31.3e} {flat[i]:.3e}")    print("  wer bei jeder Schicht die Eingabe wieder addiert, lässt")    print("  die Werte selbst anwachsen. Kapitel 21 hält das auf")

In [ ]:
main()

Das Kapitel, aus dem dieses Notebook stammt, rechnet jede Zahl oben durch und erklärt, warum jeder Schritt da ist.